In [1]:
from aalpy.learning_algs import run_Lstar
from aalpy.oracles import StatePrefixEqOracle
from aalpy.utils import save_automaton_to_file, visualize_automaton
import os
os.chdir("../scripts/extraction")

In [2]:
from DataProcessing import parse_data, preprocess_binary_classification_data
from RNN_SULs import RnnBinarySUL
from RNNClassifier import RNNClassifier

from TrainAndExtract import train_RNN_on_tomita_grammar, train_and_extract_bp, train_RNN_and_extract_FSM
import pytz, json
from datetime import datetime
timezone = pytz.timezone('America/New_York') 

[dynet] random seed: 3191478372
[dynet] allocating memory: 2048MB
[dynet] memory allocation done.


In [183]:
# Train the RNN on the data set generated from Tomita 3 Grammar
alphabet = ["(", ")"]
# load training data from file
dataset_handle = "0327_001213"
metadata = json.load(open(f"../../data/{dataset_handle}/data.json"))['metadata']
max_depth = metadata["max_depth"]

config = {
    "data_txt_file": f"../../data/{dataset_handle}/dyck1-{max_depth}.txt",
    "num_layers": 1,
    "hidden_dim": 64,
    "batch_size": 32,
    "nn_type": "GRU",
    "metadata": metadata,
}
x, y = parse_data(config["data_txt_file"])
x_train, y_train, x_test, y_test = preprocess_binary_classification_data(x, y, alphabet)
rnn = RNNClassifier(alphabet, output_dim=2, num_layers=config["num_layers"], hidden_dim=config["hidden_dim"], batch_size=config["hidden_dim"],
                    x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test, nn_type=config["nn_type"])


In [185]:
# Train the RNN
date = datetime.now(timezone).strftime("%m%d_%H%M%S")

rnn.train(stop_acc=1.0, stop_loss=0.0005)

Starting train
Epoch 0: Accuracy 1.0, Avg. Loss 0.00047 Validation Accuracy 0.99929
Done training!


In [186]:
# Wrap RNN in the SUL class. 
sul = RnnBinarySUL(rnn)

# Define the eq. oracle
state_eq_oracle = StatePrefixEqOracle(alphabet, sul, walks_per_state=200, walk_len=6)

In [187]:
# Extract the model from RNN
# Max. number of rounds is limited to be able to visualize small/correct automata.
# If it is not set adversarial inputs will be found :D 
dfa = run_Lstar(alphabet=alphabet, sul=sul, eq_oracle=state_eq_oracle, automaton_type='dfa',
                cache_and_non_det_check=True, max_learning_rounds=3)


Hypothesis 1: 1 states.
Hypothesis 2: 3 states.
Hypothesis 3: 4 states.
-----------------------------------
Learning Finished.
Learning Rounds:  3
Number of states: 4
Time (in seconds)
  Total                : 0.08
  Learning algorithm   : 0.0
  Conformance checking : 0.08
Learning Algorithm
 # Membership Queries  : 15
 # MQ Saved by Caching : 19
 # Steps               : 43
Equivalence Query
 # Membership Queries  : 800
 # Steps               : 5594
-----------------------------------


In [188]:
save_dir = f'RNN_Models/dyck1-{max_depth}/{dataset_handle}/{date}'
os.makedirs(save_dir, exist_ok=True)
save_automaton_to_file(dfa, os.path.join(save_dir, 'dfa.dot'))
json.dump(config, open(os.path.join(save_dir, 'config.json'), 'w'), indent=4)
#visualize_automaton(dfa)

Model saved to RNN_Models/dyck1-1/0327_001213/0327_020800/dfa.dot.


## Draft

In [15]:
dfa.states

In [13]:
import pydot
